# HNSW + 在线 HMM 检索与 Recall@25 评估

使用真实 NetVLAD 数据：多轨迹 Query，每条轨迹按帧做 HNSW Top-K 检索后经 OnlineHMM 时序平滑；
GT 采用坐标法（uav_infos.csv 经纬度 + 方圆 25 张），评估 Recall@25，对比「仅 HNSW」与「HNSW+HMM」。

In [217]:
import importlib
from pathlib import Path
import sys

_root = Path().resolve()
if _root.name == "hnsw_performance_analysis":
    _root = _root.parent
sys.path.insert(0, str(_root))

import numpy as np
import faiss

FPS = 4.0
FRAME_INTERVAL = 1.0 / FPS
EVAL_TOPK = 20
SAVE_TOPK = 10
WINDOW_SIZE = 3
EXPANSION_RADIUS_SCALE = 1.8
EXPANSION_RADIUS_FALLBACK_PX = 24.0
MAX_EXPANSION_CANDIDATES = 40
MAX_STATE_CANDIDATES = 40
USE_HMM_TOP1_FUSION = False
FUSION_HNSW_TOPN = 5
SPATIAL_RERANK_TOPN = 2
SPATIAL_RERANK_LAMBDA = 0.05

# 新增：基于预测位置的软惩罚（不剔除候选，只重排）
ENABLE_SOFT_SPATIAL_PENALTY = True
SOFT_PENALTY_REF_PX = 250.0   # 距离归一化参考尺度（像素）
SOFT_PENALTY_WEIGHT = 0.001    # 惩罚强度，越大越偏向近处

from demo_readH5 import (
    load_netvlad_descriptors,
    load_multi_netvlad_descriptors,
    DB_H5_PATHS,
    QUERY_H5_PATHS,
)
from scene_config import (
    get_db_scene_ranges,
    get_query_trajectory_ranges,
    get_scene_paths,
    DATASETS_BASE,
)
import coords_gt_utils
importlib.reload(coords_gt_utils)
from coords_gt_utils import (
    build_database_coords,
    build_gt_9_for_all_trajectories,
    get_geotransform_and_srs,
    lonlat_to_pixel,
    load_uav_infos,
)
from HMM.HMM import (
    OnlineHMM,
    WindowViterbiHMM,
    DEFAULT_ALPHA_SIGMA,
    DEFAULT_EMISSION_SCALE,
    DEFAULT_WINDOW_SIZE,
)

WINDOW_SIZE = max(WINDOW_SIZE, DEFAULT_WINDOW_SIZE)
print(
    f"HMM 参数: alpha_sigma={DEFAULT_ALPHA_SIGMA}, emission_scale={DEFAULT_EMISSION_SCALE}, "
    f"window_size={WINDOW_SIZE}, expansion_radius_scale={EXPANSION_RADIUS_SCALE}, "
    f"expansion_radius_fallback_px={EXPANSION_RADIUS_FALLBACK_PX}, max_state_candidates={MAX_STATE_CANDIDATES}, "
    f"use_top1_fusion={USE_HMM_TOP1_FUSION}, fusion_hnsw_topn={FUSION_HNSW_TOPN}, "
    f"spatial_rerank_topn={SPATIAL_RERANK_TOPN}, spatial_rerank_lambda={SPATIAL_RERANK_LAMBDA}"
)


def expand_candidates_with_radius(
    base_indices,
    base_distances,
    query_desc,
    db_descs,
    coords,
    db_start,
    db_end,
    anchor_idx=None,
    displacement=None,
    expansion_radius_px=None,
    max_expansion_candidates=MAX_EXPANSION_CANDIDATES,
    max_candidates=MAX_STATE_CANDIDATES,
):
    base_indices = np.asarray(base_indices, dtype=np.int64)
    base_distances = np.asarray(base_distances, dtype=np.float64)
    if base_indices.size == 0:
        return base_indices, base_distances, 0

    merged_pairs = [(int(idx), float(dist)) for idx, dist in zip(base_indices, base_distances)]
    if coords is None or anchor_idx is None or not (db_start <= int(anchor_idx) < db_end):
        return base_indices[:max_candidates], base_distances[:max_candidates], 0

    if expansion_radius_px is None or not np.isfinite(expansion_radius_px) or expansion_radius_px <= 0:
        expansion_radius_px = EXPANSION_RADIUS_FALLBACK_PX

    anchor_coord = coords[int(anchor_idx)]
    if np.any(np.isnan(anchor_coord)):
        return base_indices[:max_candidates], base_distances[:max_candidates], 0

    center = np.array(anchor_coord, dtype=np.float64)
    if displacement is not None:
        center = center + np.asarray(displacement, dtype=np.float64)

    scene_coords = coords[db_start:db_end]
    valid_mask = ~np.any(np.isnan(scene_coords), axis=1)
    if not np.any(valid_mask):
        return base_indices[:max_candidates], base_distances[:max_candidates], 0

    scene_global_indices = np.arange(db_start, db_end, dtype=np.int64)[valid_mask]
    scene_valid_coords = scene_coords[valid_mask]
    coord_dists = np.linalg.norm(scene_valid_coords - center, axis=1)
    within_radius = coord_dists <= float(expansion_radius_px)
    if not np.any(within_radius):
        return base_indices[:max_candidates], base_distances[:max_candidates], 0

    candidate_pool = list(zip(scene_global_indices[within_radius], coord_dists[within_radius]))
    candidate_pool.sort(key=lambda x: x[1])

    base_set = {int(idx) for idx in base_indices}
    extras = []
    for candidate_idx, _ in candidate_pool:
        candidate_idx = int(candidate_idx)
        if candidate_idx in base_set:
            continue
        extras.append(candidate_idx)
        if len(extras) >= max_expansion_candidates:
            break

    if not extras:
        return base_indices[:max_candidates], base_distances[:max_candidates], 0

    query_desc = np.asarray(query_desc, dtype=np.float32)
    extra_indices_arr = np.asarray(extras, dtype=np.int64)
    extra_scores = db_descs[extra_indices_arr] @ query_desc
    extra_distances = 1.0 - extra_scores.astype(np.float64)

    merged_pairs.extend((int(idx), float(dist)) for idx, dist in zip(extra_indices_arr, extra_distances))
    merged_pairs.sort(key=lambda x: x[1])

    deduped = []
    seen = set()
    for idx, dist in merged_pairs:
        if idx in seen:
            continue
        seen.add(idx)
        deduped.append((idx, dist))
        if len(deduped) >= max_candidates:
            break

    merged_indices = np.asarray([idx for idx, _ in deduped], dtype=np.int64)
    merged_distances = np.asarray([dist for _, dist in deduped], dtype=np.float64)
    extra_added = max(0, len(merged_indices) - len(base_indices))
    return merged_indices, merged_distances, extra_added


def apply_spatial_soft_penalty(
    candidate_indices,
    candidate_distances,
    coords,
    predicted_center,
    ref_px=SOFT_PENALTY_REF_PX,
    weight=SOFT_PENALTY_WEIGHT,
):
    """按到预测中心的距离给 soft penalty；不删除候选。"""
    inds = np.asarray(candidate_indices, dtype=np.int64)
    base_dists = np.asarray(candidate_distances, dtype=np.float64)
    debug = {
        "enabled": bool(ENABLE_SOFT_SPATIAL_PENALTY),
        "count": int(len(inds)),
        "applied": False,
        "mean_penalty": 0.0,
        "max_penalty": 0.0,
    }

    if (not ENABLE_SOFT_SPATIAL_PENALTY) or coords is None or predicted_center is None or len(inds) == 0:
        return base_dists, debug

    cand_coords = coords[inds]
    valid_mask = ~np.any(np.isnan(cand_coords), axis=1)
    if not np.any(valid_mask):
        return base_dists, debug

    center = np.asarray(predicted_center, dtype=np.float64)
    spatial_d = np.zeros(len(inds), dtype=np.float64)
    spatial_d.fill(float(ref_px))
    spatial_d[valid_mask] = np.linalg.norm(cand_coords[valid_mask] - center[None, :], axis=1)

    ref = max(float(ref_px), 1e-6)
    penalty = float(weight) * (spatial_d / ref) ** 2
    out_dists = base_dists + penalty

    debug["applied"] = True
    debug["mean_penalty"] = float(np.mean(penalty))
    debug["max_penalty"] = float(np.max(penalty))
    debug["ref_px"] = float(ref_px)
    return out_dists, debug


def rerank_candidates_by_predicted_center(
    ranked_indices,
    coords,
    predicted_center,
    rerank_topn=SPATIAL_RERANK_TOPN,
    fusion_lambda=SPATIAL_RERANK_LAMBDA,
):
    ranked_indices = [int(x) for x in ranked_indices]
    if len(ranked_indices) <= 1 or coords is None or predicted_center is None:
        return ranked_indices, None

    topn = min(int(rerank_topn), len(ranked_indices))
    top_indices = ranked_indices[:topn]
    top_coords = coords[np.asarray(top_indices, dtype=np.int64)]
    if np.any(np.isnan(top_coords)):
        return ranked_indices, None

    predicted_center = np.asarray(predicted_center, dtype=np.float64)
    dists = np.linalg.norm(top_coords - predicted_center[None, :], axis=1)
    hmm_rank_score = np.linspace(1.0, 0.0, topn, endpoint=True) if topn > 1 else np.array([1.0])
    max_dist = float(np.max(dists))
    min_dist = float(np.min(dists))
    if max_dist - min_dist < 1e-9:
        spatial_score = np.ones(topn, dtype=np.float64)
    else:
        spatial_score = 1.0 - (dists - min_dist) / (max_dist - min_dist)

    combined_score = fusion_lambda * hmm_rank_score + (1.0 - fusion_lambda) * spatial_score
    order = np.argsort(-combined_score)
    reranked_top = [top_indices[i] for i in order]
    reranked = reranked_top + ranked_indices[topn:]
    debug = {
        "top_indices": top_indices,
        "distances": dists.tolist(),
        "spatial_score": spatial_score.tolist(),
        "hmm_rank_score": hmm_rank_score.tolist(),
        "combined_score": combined_score.tolist(),
    }
    return reranked, debug

HMM 参数: alpha_sigma=0.3, emission_scale=5.0, window_size=3, expansion_radius_scale=1.8, expansion_radius_fallback_px=24.0, max_state_candidates=40, use_top1_fusion=False, fusion_hnsw_topn=5, spatial_rerank_topn=2, spatial_rerank_lambda=0.05


## 1. 加载 DB 与 Query（多轨迹）

In [197]:
print("加载 DB...")
db_names, db_descs = load_multi_netvlad_descriptors(DB_H5_PATHS)
db_descs = db_descs.astype(np.float32)
N_db, D = db_descs.shape
print(f"DB: {N_db} 条, 维度 {D}")

print("按轨迹加载 Query...")
query_per_traj = []
for h5_path in QUERY_H5_PATHS:
    names, descs = load_netvlad_descriptors(Path(h5_path))
    query_per_traj.append((names, descs.astype(np.float32)))

query_names = []
query_descs_list = []
for _, (names, descs) in enumerate(query_per_traj):
    query_names.extend(names)
    query_descs_list.append(descs)
query_descs = np.vstack(query_descs_list)
trajectory_ranges = get_query_trajectory_ranges(QUERY_H5_PATHS)
n_queries = query_descs.shape[0]
print(f"Query: {n_queries} 条, 共 {len(trajectory_ranges)} 条轨迹")
db_scene_ranges = get_db_scene_ranges(DB_H5_PATHS, db_names)

加载 DB...
DB: 11009 条, 维度 4096
按轨迹加载 Query...
Query: 5988 条, 共 15 条轨迹


## 2. DB 坐标（供 HMM 转移约束）与坐标法 GT@25

In [198]:
print("构建 database_coords（GDAL + id_startx_starty）...")
database_coords = build_database_coords(db_names, db_scene_ranges, DATASETS_BASE)
n_valid = np.sum(~np.any(np.isnan(database_coords), axis=1))
print(f"有效坐标: {n_valid}/{N_db}")

print("构建坐标法 GT@25（每 query 方圆 25 张）...")
gt_9_list, gt_25_ordered = build_gt_9_for_all_trajectories(
    trajectory_ranges,
    db_scene_ranges,
    db_names,
    DATASETS_BASE,
)
n_with_gt = sum(1 for s in gt_9_list if len(s) > 0)
print(f"有 GT 的 query 数: {n_with_gt}/{len(gt_9_list)}")
if n_with_gt > 0:
    sizes = [len(s) for s in gt_9_list if len(s) > 0]
    print(f"GT 集合大小: min={min(sizes)}, max={max(sizes)}, mean={sum(sizes)/len(sizes):.1f}  (若 max>9 说明已按 25 张生成)")
    i0 = next(i for i in range(len(gt_9_list)) if len(gt_9_list[i]) > 0)
    try:
        if i0 < len(gt_25_ordered) and gt_25_ordered[i0] and isinstance(gt_25_ordered[i0][0], (int, np.integer)):
            gt0 = list(gt_25_ordered[i0])
        else:
            gt0 = sorted(gt_9_list[i0])
    except NameError:
        gt0 = sorted(gt_9_list[i0])
    print(f"\n第一张有 GT 的 query（index={i0}）的 25 张 GT 图片:")
    for j, idx in enumerate(gt0):
        idx = int(idx) if not isinstance(idx, (int, np.integer)) else idx
        name = db_names[idx] if idx < len(db_names) else f"<{idx}>"
        print(f"  [{j+1:2d}] {name}")

构建 database_coords（GDAL + id_startx_starty）...
=== database_coords 各场景（未算出坐标的会打印原因）===
  [city1] 共 342 条 -> 有效坐标 342/342
  [city2] 共 168 条 -> 有效坐标 168/168
  [city3] 共 224 条 -> 有效坐标 224/224
  [industry1] 共 868 条 -> 有效坐标 868/868
  [industry2] 共 414 条 -> 有效坐标 414/414
  [industry3] 共 1476 条 -> 有效坐标 1476/1476
  [park1] 共 324 条 -> 有效坐标 324/324
  [rural1] 共 1845 条 -> 有效坐标 1845/1845
  [rural2] 共 1845 条 -> 有效坐标 1845/1845
  [rural3] 共 399 条 -> 有效坐标 399/399
  [school] 共 1178 条 -> 有效坐标 1178/1178
  [suburbs1] 共 480 条 -> 有效坐标 480/480
  [suburbs2] 共 468 条 -> 有效坐标 468/468
  [village1] 共 285 条 -> 有效坐标 285/285
  [village2] 共 693 条 -> 有效坐标 693/693
  -> 合计有效坐标: 11009/11009

有效坐标: 11009/11009
构建坐标法 GT@25（每 query 方圆 25 张）...
=== 坐标法 GT@25 各轨迹（未算出 GT 的会打印原因）===

--- 坐标法 GT 首帧诊断 [city1] ---
大图 mapbox geotransform (6 参数):
  gt[0] 左上角 X (投影/经度): 12123218.434142068
  gt[1] 像元宽:               0.2985821417389691
  gt[2] 旋转(常为 0):         0.0
  gt[3] 左上角 Y (投影/纬度): 4062398.742272254
  gt[4] 旋转(常为 0):         0.0
  

## 3. HNSW 索引（NetVLAD 已归一化，用内积）

In [199]:
K = 30
M = 32
ef_construction = 240
ef_search = 80

# 内积索引（归一化向量上等价于余弦）；Faiss 内积返回越大越相似
index = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search
index.add(db_descs)
print(f"HNSW 索引已构建: M={M}, efConstruction={ef_construction}, efSearch={ef_search}, K={K}")

HNSW 索引已构建: M=32, efConstruction=240, efSearch=80, K=30


## 4. 逐轨迹运行 HNSW + HMM，并计算 Recall@1 / @5 / @10

In [222]:
# 检索方式：按场景检索（每条 query 只在该 query 所属场景的 DB 内检索，非全库检索）
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

index_per_scene = {}
for scene_name, db_start, db_end in db_scene_ranges:
    seg = db_descs[db_start:db_end]
    idx = faiss.IndexHNSWFlat(D, M, faiss.METRIC_INNER_PRODUCT)
    idx.hnsw.efConstruction = ef_construction
    idx.hnsw.efSearch = ef_search
    idx.add(seg)
    index_per_scene[scene_name] = (idx, db_start, db_end)

pred_hnsw = np.full(n_queries, -1, dtype=np.int64)
pred_hmm = np.full(n_queries, -1, dtype=np.int64)
store_topk = min(K, EVAL_TOPK)
topk_hnsw = np.full((n_queries, store_topk), -1, dtype=np.int64)
topk_hmm = np.full((n_queries, store_topk), -1, dtype=np.int64)
query_px_all = np.full(n_queries, np.nan, dtype=np.float64)
query_py_all = np.full(n_queries, np.nan, dtype=np.float64)
fusion_used_hmm = 0
fusion_fallback_hnsw = 0
query_idx = 0

for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    idx_scene, db_start, db_end = index_per_scene.get(scene_name, (None, 0, 0))
    if idx_scene is None:
        query_idx += (end - start)
        continue
    q_descs = query_descs[start:end]
    n_frames = q_descs.shape[0]
    k_scene = min(K, db_end - db_start)
    D_t, I_local = idx_scene.search(q_descs, k_scene)
    I_t = I_local.astype(np.int64) + db_start
    dist_for_hmm = 1.0 - D_t.astype(np.float64)

    # 本轨迹 query 在大图像素坐标，用于 HMM 的真实位移/真实速度先验（4 fps）
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])
    n_use = min(len(lats), n_frames)
    query_px = [None] * n_frames
    query_py = [None] * n_frames
    frame_speeds = [None] * n_frames
    frame_displacements = []
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = px, py
    for f in range(1, n_use):
        if query_px[f - 1] is None or query_px[f] is None:
            continue
        dx = query_px[f] - query_px[f - 1]
        dy = query_py[f] - query_py[f - 1]
        d_pix = float(np.hypot(dx, dy))
        frame_displacements.append(d_pix)
        frame_speeds[f] = d_pix * FPS
    valid_speeds = [s for s in frame_speeds if s is not None and np.isfinite(s) and s > 0]
    traj_mean_speed = float(np.mean(valid_speeds)) if valid_speeds else None
    if frame_displacements:
        disp_arr = np.asarray(frame_displacements, dtype=np.float64)
        median_disp = float(np.median(disp_arr))
        expansion_radius_px = max(EXPANSION_RADIUS_FALLBACK_PX, EXPANSION_RADIUS_SCALE * median_disp)
        print(
            f"[{scene_name}] 真实帧间像素位移统计: "
            f"mean={disp_arr.mean():.2f}px  median={median_disp:.2f}px  "
            f"p90={np.percentile(disp_arr, 90):.2f}px  max={disp_arr.max():.2f}px  "
            f"expansion_radius={expansion_radius_px:.2f}px  (n={len(disp_arr)})"
        )
    else:
        median_disp = None
        expansion_radius_px = EXPANSION_RADIUS_FALLBACK_PX
        print(f"[{scene_name}] 真实帧间像素位移统计: 无有效位移数据, expansion_radius={expansion_radius_px:.2f}px")

    verbose_hmm = (traj_idx == 0)
    if verbose_hmm:
        print("========== 第一个场景改进 HMM 过程（逐帧）==========")
        print(
            f"  window_size={WINDOW_SIZE}, expansion_radius_scale={EXPANSION_RADIUS_SCALE}, "
            f"expansion_radius_px={expansion_radius_px:.2f}, max_state_candidates={MAX_STATE_CANDIDATES}"
        )
        if traj_mean_speed is not None:
            print(f"  轨迹平均真实速度 = {traj_mean_speed:.4f} pixel/s (fps={FPS})")
        else:
            print(f"  未从真实坐标估计出有效速度，fps={FPS}，将退回到位移/均匀转移")
    hmm = WindowViterbiHMM(
        coords_for_hmm,
        MAX_STATE_CANDIDATES,
        uav_speed=traj_mean_speed,
        delta_t=FRAME_INTERVAL,
        window_size=WINDOW_SIZE,
        verbose=verbose_hmm,
    )
    for f in range(n_frames):
        if query_px[f] is not None and query_py[f] is not None:
            query_px_all[query_idx] = float(query_px[f])
            query_py_all[query_idx] = float(query_py[f])
        pred_hnsw[query_idx] = I_t[f, 0]
        n_top = min(store_topk, I_t.shape[1])
        topk_hnsw[query_idx, :n_top] = I_t[f, :n_top]

        base_inds = I_t[f]
        base_dists = dist_for_hmm[f]
        displacement = None
        frame_speed = frame_speeds[f]
        if f >= 1 and query_px[f - 1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f - 1], query_py[f] - query_py[f - 1])

        predicted_center = None
        # 按你的要求：预测中心 = 上一帧 query 真实坐标 + 当前位移
        if f >= 1 and query_px[f - 1] is not None and query_py[f - 1] is not None and displacement is not None:
            predicted_center = np.array(
                [query_px[f - 1] + displacement[0], query_py[f - 1] + displacement[1]],
                dtype=np.float64,
            )
        # 首帧或位移缺失时，退化为当前帧真实坐标
        elif query_px[f] is not None and query_py[f] is not None:
            predicted_center = np.array([query_px[f], query_py[f]], dtype=np.float64)

        # 新增：对候选做空间软惩罚（距离预测位置越远，代价越高）
        penalized_base_dists, penalty_debug_base = apply_spatial_soft_penalty(
            base_inds,
            base_dists,
            coords_for_hmm,
            predicted_center,
        )

        expanded_inds, expanded_dists, extra_added = expand_candidates_with_radius(
            base_inds,
            penalized_base_dists,
            q_descs[f],
            db_descs,
            coords_for_hmm,
            db_start,
            db_end,
            anchor_idx=hmm.prev_best_global,
            displacement=displacement,
            expansion_radius_px=expansion_radius_px,
        )
        # 扩展后再做一次 soft penalty，再送入 HMM（等价于在 emission 中加入空间先验）
        penalized_expanded_dists, penalty_debug_expanded = apply_spatial_soft_penalty(
            expanded_inds,
            expanded_dists,
            coords_for_hmm,
            predicted_center,
        )
        hmm_top, window_debug = hmm.add_frame(
            expanded_inds,
            penalized_expanded_dists,
            return_top_k=store_topk,
            displacement=displacement,
            frame_speed=frame_speed,
            metadata={"extra_neighbors": extra_added, "frame_index": f},
            return_debug=True,
        )
        hmm_top_reranked, rerank_debug = rerank_candidates_by_predicted_center(
            hmm_top,
            coords_for_hmm,
            predicted_center,
        )
        if verbose_hmm and (f < 5 or extra_added > 0):
            if penalty_debug_expanded.get("applied", False):
                print(
                    f"  [soft-penalty] frame={f} mean={penalty_debug_expanded.get('mean_penalty', 0.0):.4f} "
                    f"max={penalty_debug_expanded.get('max_penalty', 0.0):.4f} ref_px={penalty_debug_expanded.get('ref_px', SOFT_PENALTY_REF_PX)}"
                )
            print(
                f"  [expand] frame={f} base_candidates={len(base_inds)} "
                f"expanded_candidates={len(expanded_inds)} extra_candidates={extra_added} "
                f"radius_px={expansion_radius_px:.2f} window_length={window_debug['window_length']}"
            )
            if rerank_debug is not None:
                print(
                    f"  [rerank] frame={f} top_indices={rerank_debug['top_indices']} "
                    f"distances={[round(x, 2) for x in rerank_debug['distances']]}"
                )

        hnsw_ranked = [int(x) for x in base_inds[:store_topk]]
        fused_ranking = [int(x) for x in hmm_top_reranked[:store_topk]] if hmm_top_reranked else hnsw_ranked.copy()
        warm_start_active = (f < 3 and len(gt_9_list[query_idx]) > 0)
        if USE_HMM_TOP1_FUSION and not warm_start_active and hmm_top_reranked:
            hnsw_topn = {int(x) for x in base_inds[: min(FUSION_HNSW_TOPN, len(base_inds))]}
            hmm_top1 = int(hmm_top_reranked[0])
            if hmm_top1 in hnsw_topn:
                fusion_used_hmm += 1
                fused_top1 = hmm_top1
                fusion_source = "HMM"
            else:
                fusion_fallback_hnsw += 1
                fused_top1 = int(base_inds[0])
                fusion_source = "HNSW"
            fused_ranking = [fused_top1]
            fused_ranking.extend(int(x) for x in hmm_top_reranked if int(x) != fused_top1)
            fused_ranking.extend(int(x) for x in hnsw_ranked if int(x) != fused_top1)
            deduped_ranking = []
            seen = set()
            for idx in fused_ranking:
                if idx in seen:
                    continue
                seen.add(idx)
                deduped_ranking.append(idx)
                if len(deduped_ranking) >= store_topk:
                    break
            fused_ranking = deduped_ranking
            if verbose_hmm and (f < 5 or fusion_source == "HNSW"):
                print(
                    f"  [fusion] frame={f} source={fusion_source} "
                    f"hmm_top1={hmm_top1} hnsw_top1={int(base_inds[0])} "
                    f"hnsw_top{FUSION_HNSW_TOPN}={list(base_inds[: min(FUSION_HNSW_TOPN, len(base_inds))])}"
                )
        pred_hmm[query_idx] = fused_ranking[0] if fused_ranking else -1
        topk_hmm[query_idx, :] = -1
        for j, idx in enumerate(fused_ranking[:store_topk]):
            topk_hmm[query_idx, j] = idx

        # 每场景前 3 帧用 GT 的 top1 做 warm-start，下一帧窗口解码将从该 GT 重新锚定
        if warm_start_active:
            gt_idx = gt_25_ordered[query_idx][0] if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else min(gt_9_list[query_idx])
            if verbose_hmm:
                gt_name = db_names[gt_idx] if 0 <= gt_idx < len(db_names) else "?"
                print(f"  [warm-start] 帧 f={f} 用 GT top1 替代: gt_idx={gt_idx} -> {gt_name}")
            pred_hmm[query_idx] = gt_idx
            topk_hmm[query_idx, :] = -1
            topk_hmm[query_idx, 0] = gt_idx
            rest = [x for x in fused_ranking if x != gt_idx][: max(0, store_topk - 1)]
            for j, x in enumerate(rest):
                topk_hmm[query_idx, j + 1] = x
            hmm.override_prev_best(gt_idx)
        query_idx += 1

assert query_idx == n_queries
print(
    f"Top1 融合统计: 使用 HMM top1 {fusion_used_hmm} 次, "
    f"回退到 HNSW top1 {fusion_fallback_hnsw} 次, "
    f"融合启用={USE_HMM_TOP1_FUSION}, HNSW topN={FUSION_HNSW_TOPN}"
)

[city1] 真实帧间像素位移统计: mean=16.24px  median=15.58px  p90=18.64px  max=21.23px  expansion_radius=28.04px  (n=428)
========== 第一个场景改进 HMM 过程（逐帧）==========
  window_size=3, expansion_radius_scale=1.8, expansion_radius_px=28.04, max_state_candidates=40
  轨迹平均真实速度 = 64.9603 pixel/s (fps=4.0)

--- Window HMM Frame 0 ---
  window_length = 1/3
  candidate_counts = [30]
  extra_neighbors = 0
  displacement = None
  frame_speed = None
  Window HMM 输出 top-1 = 175, top-3 = [175, 174, 196]
  best_path = [175]
  [soft-penalty] frame=0 mean=0.0042 max=0.0341 ref_px=250.0
  [expand] frame=0 base_candidates=30 expanded_candidates=30 extra_candidates=0 radius_px=28.04 window_length=1
  [rerank] frame=0 top_indices=[175, 174] distances=[97.37, 176.83]
  [warm-start] 帧 f=0 用 GT top1 替代: gt_idx=196 -> tif/278_1906_2356.tif
    [transition] prev_best_global=196, center=[2275.08691912 2744.51009573], motion_radius=15.5801, radius=31.2, sigma=138.6512, displacement=(-14.913080883758084, 4.51009573291185), frame_

In [223]:
valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)])
n_eval = int(np.sum(valid))

topk_eval_5 = min(5, topk_hnsw.shape[1])
topk_eval_10 = min(10, topk_hnsw.shape[1])
topk_eval_20 = min(20, topk_hnsw.shape[1])

# Recall@1 / @5 / @10 / @20：top-k 中是否有任一张在 GT（方圆 25 张）内
correct_1_hnsw = np.array([topk_hnsw[i, 0] in gt_9_list[i] if topk_hnsw[i, 0] >= 0 else False for i in range(n_queries)])
correct_5_hnsw = np.array([any(topk_hnsw[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
correct_10_hnsw = np.array([any(topk_hnsw[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])
correct_20_hnsw = np.array([any(topk_hnsw[i, j] in gt_9_list[i] for j in range(topk_eval_20)) for i in range(n_queries)])
correct_1_hmm = np.array([pred_hmm[i] in gt_9_list[i] for i in range(n_queries)])
correct_5_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
correct_10_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])
correct_20_hmm = np.array([any(topk_hmm[i, j] in gt_9_list[i] for j in range(topk_eval_20)) for i in range(n_queries)])

center_gt_idx = np.full(n_queries, -1, dtype=np.int64)
for i in range(n_queries):
    if i < len(gt_25_ordered) and gt_25_ordered[i]:
        center_gt_idx[i] = int(gt_25_ordered[i][0])
    elif len(gt_9_list[i]) > 0:
        center_gt_idx[i] = int(min(gt_9_list[i]))

center_acc_hnsw = np.array([
    topk_hnsw[i, 0] == center_gt_idx[i] if center_gt_idx[i] >= 0 and topk_hnsw[i, 0] >= 0 else False
    for i in range(n_queries)
])
center_acc_hmm = np.array([
    pred_hmm[i] == center_gt_idx[i] if center_gt_idx[i] >= 0 and pred_hmm[i] >= 0 else False
    for i in range(n_queries)
])

query_coords_valid = np.isfinite(query_px_all) & np.isfinite(query_py_all)

def _top1_localization_errors(pred_indices):
    errors = np.full(n_queries, np.nan, dtype=np.float64)
    for i in range(n_queries):
        pred_idx = int(pred_indices[i])
        if pred_idx < 0 or pred_idx >= len(database_coords):
            continue
        if not query_coords_valid[i]:
            continue
        pred_coord = database_coords[pred_idx]
        if np.any(np.isnan(pred_coord)):
            continue
        errors[i] = float(np.hypot(pred_coord[0] - query_px_all[i], pred_coord[1] - query_py_all[i]))
    return errors

loc_error_hnsw = _top1_localization_errors(topk_hnsw[:, 0])
loc_error_hmm = _top1_localization_errors(pred_hmm)

# 诊断：query 与预测分别属于哪个场景（全局检索时预测常落在其它场景导致 Recall=0）
query_to_scene = {}
for scene_name, start, end in trajectory_ranges:
    for i in range(start, end):
        query_to_scene[i] = scene_name
db_idx_to_scene = {}
for scene_name, start, end in db_scene_ranges:
    for j in range(start, end):
        db_idx_to_scene[j] = scene_name
valid_indices = np.where(valid)[0]
n_diag = min(5, len(valid_indices))
print("诊断（前 %d 条有 GT 的 query）：" % n_diag)
for k in range(n_diag):
    i = valid_indices[k]
    q_scene = query_to_scene.get(i, "?")
    p_hnsw = int(pred_hnsw[i])
    p_hmm = int(pred_hmm[i])
    p_hnsw_scene = db_idx_to_scene.get(p_hnsw, "?")
    p_hmm_scene = db_idx_to_scene.get(p_hmm, "?")
    gt_set = (gt_25_ordered[i][:5] if i < len(gt_25_ordered) and gt_25_ordered[i] else list(gt_9_list[i])[:5])
    in_hnsw = correct_1_hnsw[i]
    in_hmm = correct_1_hmm[i]
    print(f"  query {i}: 场景={q_scene} | HNSW pred={p_hnsw} (场景={p_hnsw_scene}) in_GT={in_hnsw} | HMM pred={p_hmm} (场景={p_hmm_scene}) in_GT={in_hmm} | GT 示例={gt_set}")
same_scene_hnsw = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hnsw[i]))
same_scene_hmm = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hmm[i]))
print(f"预测与 query 同场景的比例: HNSW {same_scene_hnsw}/{n_eval}, HMM {same_scene_hmm}/{n_eval}")
i0 = valid_indices[0]
p0 = int(pred_hnsw[i0])
gt0 = gt_25_ordered[i0] if (i0 < len(gt_25_ordered) and gt_25_ordered[i0]) else list(gt_9_list[i0])
print("")
print("query 0 细查（看预测与 GT 子图是否相邻）:")
print(f"  预测子图 db_names[{p0}] = {db_names[p0]}")
for g in sorted(gt0)[:5]:
    print(f"  GT 子图 db_names[{g}] = {db_names[g]}")
print("  （若 id_startx_starty 相差很大，说明坐标→瓦片或 query 与 uav_infos 行序可能不一致）")
print("前 3 条 query 的图像名（请与 uav_infos 前 3 行的 file_name 核对是否一一对应）:")
for k in range(min(3, len(query_names))):
    print(f"  query {k}: {query_names[k]}")
print("")

if n_eval > 0:
    r1_hnsw = correct_1_hnsw[valid].mean()
    r5_hnsw = correct_5_hnsw[valid].mean()
    r10_hnsw = correct_10_hnsw[valid].mean()
    r20_hnsw = correct_20_hnsw[valid].mean()
    r1_hmm = correct_1_hmm[valid].mean()
    r5_hmm = correct_5_hmm[valid].mean()
    r10_hmm = correct_10_hmm[valid].mean()
    r20_hmm = correct_20_hmm[valid].mean()
    center_valid = valid & (center_gt_idx >= 0)
    center_acc1_hnsw = center_acc_hnsw[center_valid].mean() if np.any(center_valid) else np.nan
    center_acc1_hmm = center_acc_hmm[center_valid].mean() if np.any(center_valid) else np.nan
    loc_valid_hnsw = valid & np.isfinite(loc_error_hnsw)
    loc_valid_hmm = valid & np.isfinite(loc_error_hmm)
    median_loc_hnsw = np.median(loc_error_hnsw[loc_valid_hnsw]) if np.any(loc_valid_hnsw) else np.nan
    median_loc_hmm = np.median(loc_error_hmm[loc_valid_hmm]) if np.any(loc_valid_hmm) else np.nan
    mean_loc_hnsw = np.mean(loc_error_hnsw[loc_valid_hnsw]) if np.any(loc_valid_hnsw) else np.nan
    mean_loc_hmm = np.mean(loc_error_hmm[loc_valid_hmm]) if np.any(loc_valid_hmm) else np.nan
    print("Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  HNSW:        Recall@1={r1_hnsw:.4f}  Recall@5={r5_hnsw:.4f}  Recall@10={r10_hnsw:.4f}  Recall@20={r20_hnsw:.4f}  (n={n_eval})")
    print(f"  HNSW + HMM:  Recall@1={r1_hmm:.4f}  Recall@5={r5_hmm:.4f}  Recall@10={r10_hmm:.4f}  Recall@20={r20_hmm:.4f}")
    print("附加指标:")
    print(f"  HNSW:        Center-tile Accuracy@1={center_acc1_hnsw:.4f}  Median localization error={median_loc_hnsw:.2f}px  Mean localization error={mean_loc_hnsw:.2f}px")
    print(f"  HNSW + HMM:  Center-tile Accuracy@1={center_acc1_hmm:.4f}  Median localization error={median_loc_hmm:.2f}px  Mean localization error={mean_loc_hmm:.2f}px")
    print("")
    print("各场景 Recall（仅统计该场景中有坐标法 GT 的 query）:")
    for scene_name, start, end in trajectory_ranges:
        scene_indices = np.arange(start, end)
        scene_valid = valid[scene_indices]
        n_scene_eval = int(np.sum(scene_valid))
        if n_scene_eval == 0:
            print(f"  [{scene_name}] 无有效 GT，跳过")
            continue
        r1_hnsw_scene = correct_1_hnsw[scene_indices][scene_valid].mean()
        r5_hnsw_scene = correct_5_hnsw[scene_indices][scene_valid].mean()
        r10_hnsw_scene = correct_10_hnsw[scene_indices][scene_valid].mean()
        r20_hnsw_scene = correct_20_hnsw[scene_indices][scene_valid].mean()
        r1_hmm_scene = correct_1_hmm[scene_indices][scene_valid].mean()
        r5_hmm_scene = correct_5_hmm[scene_indices][scene_valid].mean()
        r10_hmm_scene = correct_10_hmm[scene_indices][scene_valid].mean()
        r20_hmm_scene = correct_20_hmm[scene_indices][scene_valid].mean()
        scene_center_valid = center_valid[scene_indices]
        scene_loc_valid_hnsw = loc_valid_hnsw[scene_indices]
        scene_loc_valid_hmm = loc_valid_hmm[scene_indices]
        center_hnsw_scene = center_acc_hnsw[scene_indices][scene_center_valid].mean() if np.any(scene_center_valid) else np.nan
        center_hmm_scene = center_acc_hmm[scene_indices][scene_center_valid].mean() if np.any(scene_center_valid) else np.nan
        median_loc_hnsw_scene = np.median(loc_error_hnsw[scene_indices][scene_loc_valid_hnsw]) if np.any(scene_loc_valid_hnsw) else np.nan
        median_loc_hmm_scene = np.median(loc_error_hmm[scene_indices][scene_loc_valid_hmm]) if np.any(scene_loc_valid_hmm) else np.nan
        print(f"  [{scene_name}] HNSW:       R@1={r1_hnsw_scene:.4f}  R@5={r5_hnsw_scene:.4f}  R@10={r10_hnsw_scene:.4f}  R@20={r20_hnsw_scene:.4f}  Center@1={center_hnsw_scene:.4f}  MedErr={median_loc_hnsw_scene:.2f}px  (n={n_scene_eval})")
        print(f"  [{scene_name}] HNSW+HMM:   R@1={r1_hmm_scene:.4f}  R@5={r5_hmm_scene:.4f}  R@10={r10_hmm_scene:.4f}  R@20={r20_hmm_scene:.4f}  Center@1={center_hmm_scene:.4f}  MedErr={median_loc_hmm_scene:.2f}px")
else:
    print("无有效坐标法 GT，请检查 uav_infos.csv 与 GDAL 路径。")

# 将 HNSW 与 HNSW+HMM 的 top-10 结果保存到 ch3/results（每行：查询图像名\t检索到的 db 图像名 x10）
results_dir = Path(_root) / "results"
results_dir.mkdir(parents=True, exist_ok=True)
def _name(i, names): return names[i] if 0 <= i < len(names) else "-"
with open(results_dir / "hnsw_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hnsw[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "hnsw_hmm_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hmm[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "gt_25.txt", "w", encoding="utf-8") as f:
    f.write("query\t" + "\t".join(f"db_{j}" for j in range(1, 26)) + "\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        gt_indices = gt_25_ordered[i][:25] if i < len(gt_25_ordered) and gt_25_ordered[i] else sorted(gt_9_list[i])[:25]
        db_cols = [_name(idx, db_names) for idx in gt_indices]
        db_cols += ["-"] * (25 - len(db_cols))
        f.write("\t".join([q] + db_cols) + "\n")
print(f"按场景检索 Top-10 已保存: {results_dir / 'hnsw_top10.txt'}, {results_dir / 'hnsw_hmm_top10.txt'}")
print(f"GT 已保存: {results_dir / 'gt_25.txt'}")



诊断（前 5 条有 GT 的 query）：
  query 0: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=196 (场景=city1) in_GT=True | GT 示例=[196, 175, 195, 197, 174]
  query 1: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=196 (场景=city1) in_GT=True | GT 示例=[196, 175, 195, 174, 197]
  query 2: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=196 (场景=city1) in_GT=True | GT 示例=[196, 175, 195, 174, 197]
  query 3: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=196 (场景=city1) in_GT=True | GT 示例=[196, 195, 175, 174, 217]
  query 4: 场景=city1 | HNSW pred=175 (场景=city1) in_GT=True | HMM pred=175 (场景=city1) in_GT=True | GT 示例=[196, 195, 175, 174, 217]
预测与 query 同场景的比例: HNSW 5988/5988, HMM 5988/5988

query 0 细查（看预测与 GT 子图是否相邻）:
  预测子图 db_names[175] = tif/259_1906_2206.tif
  GT 子图 db_names[133] = tif/220_1756_1906.tif
  GT 子图 db_names[134] = tif/221_1906_1906.tif
  GT 子图 db_names[152] = tif/238_1606_2056.tif
  GT 子图 db_names[153] = tif/239_1756_2056.tif
  GT 子图 db_names[155] = ti

In [ ]:
-1, dtype=np.int64)
for i in range(n_queries):
    if i < len(gt_25_ordered) and gt_25_ordered[i]:
        center_gt_idx[i] = int(gt_25_ordered[i][0])
    elif len(gt_9_list[i]) > 0:
        center_gt_idx[i] = int(min(gt_9_list[i]))

center_acc_hnsw = np.array([
    topk_hnsw[i, 0] == center_gt_idx[i] if center_gt_idx[i] >= 0 and topk_hnsw[i, 0] >= 0 else False
    for i in range(n_queries)
])
center_acc_hmm = np.array([
    pred_hmm[i] == center_gt_idx[i] if center_gt_idx[i] >= 0 and pred_hmm[i] >= 0 else False
    for i in range(n_queries)
])

query_coords_valid = np.isfinite(query_px_all) & np.isfinite(query_py_all)

def _top1_localization_errors(pred_indices):
    errors = np.full(n_queries, np.nan, dtype=np.float64)
    for i in range(n_queries):
        pred_idx = int(pred_indices[i])
        if pred_idx < 0 or pred_idx >= len(database_coords):
            continue
        if not query_coords_valid[i]:
            continue
        pred_coord = database_coords[pred_idx]
        if np.any(np.isnan(pred_coord)):
            continue
        errors[i] = float(np.hypot(pred_coord[0] - query_px_all[i], pred_coord[1] - query_py_all[i]))
    return errors

loc_error_hnsw = _top1_localization_errors(topk_hnsw[:, 0])
loc_error_hmm = _top1_localization_errors(pred_hmm)

# 诊断：query 与预测分别属于哪个场景（全局检索时预测常落在其它场景导致 Recall=0）
query_to_scene = {}
for scene_name, start, end in trajectory_ranges:
    for i in range(start, end):
        query_to_scene[i] = scene_name
db_idx_to_scene = {}
for scene_name, start, end in db_scene_ranges:
    for j in range(start, end):
        db_idx_to_scene[j] = scene_name
valid_indices = np.where(valid)[0]
n_diag = min(5, len(valid_indices))
print("诊断（前 %d 条有 GT 的 query）：" % n_diag)
for k in range(n_diag):
    i = valid_indices[k]
    q_scene = query_to_scene.get(i, "?")
    p_hnsw = int(pred_hnsw[i])
    p_hmm = int(pred_hmm[i])
    p_hnsw_scene = db_idx_to_scene.get(p_hnsw, "?")
    p_hmm_scene = db_idx_to_scene.get(p_hmm, "?")
    gt_set = (gt_25_ordered[i][:5] if i < len(gt_25_ordered) and gt_25_ordered[i] else list(gt_9_list[i])[:5])
    in_hnsw = correct_1_hnsw[i]
    in_hmm = correct_1_hmm[i]
    print(f"  query {i}: 场景={q_scene} | HNSW pred={p_hnsw} (场景={p_hnsw_scene}) in_GT={in_hnsw} | HMM pred={p_hmm} (场景={p_hmm_scene}) in_GT={in_hmm} | GT 示例={gt_set}")
same_scene_hnsw = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hnsw[i]))
same_scene_hmm = sum(1 for i in valid_indices if query_to_scene.get(i) == db_idx_to_scene.get(pred_hmm[i]))
print(f"预测与 query 同场景的比例: HNSW {same_scene_hnsw}/{n_eval}, HMM {same_scene_hmm}/{n_eval}")
i0 = valid_indices[0]
p0 = int(pred_hnsw[i0])
gt0 = gt_25_ordered[i0] if (i0 < len(gt_25_ordered) and gt_25_ordered[i0]) else list(gt_9_list[i0])
print("")
print("query 0 细查（看预测与 GT 子图是否相邻）:")
print(f"  预测子图 db_names[{p0}] = {db_names[p0]}")
for g in sorted(gt0)[:5]:
    print(f"  GT 子图 db_names[{g}] = {db_names[g]}")
print("  （若 id_startx_starty 相差很大，说明坐标→瓦片或 query 与 uav_infos 行序可能不一致）")
print("前 3 条 query 的图像名（请与 uav_infos 前 3 行的 file_name 核对是否一一对应）:")
for k in range(min(3, len(query_names))):
    print(f"  query {k}: {query_names[k]}")
print("")

if n_eval > 0:
    r1_hnsw = correct_1_hnsw[valid].mean()
    r5_hnsw = correct_5_hnsw[valid].mean()
    r10_hnsw = correct_10_hnsw[valid].mean()
    r20_hnsw = correct_20_hnsw[valid].mean()
    r1_hmm = correct_1_hmm[valid].mean()
    r5_hmm = correct_5_hmm[valid].mean()
    r10_hmm = correct_10_hmm[valid].mean()
    r20_hmm = correct_20_hmm[valid].mean()
    center_valid = valid & (center_gt_idx >= 0)
    center_acc1_hnsw = center_acc_hnsw[center_valid].mean() if np.any(center_valid) else np.nan
    center_acc1_hmm = center_acc_hmm[center_valid].mean() if np.any(center_valid) else np.nan
    loc_valid_hnsw = valid & np.isfinite(loc_error_hnsw)
    loc_valid_hmm = valid & np.isfinite(loc_error_hmm)
    median_loc_hnsw = np.median(loc_error_hnsw[loc_valid_hnsw]) if np.any(loc_valid_hnsw) else np.nan
    median_loc_hmm = np.median(loc_error_hmm[loc_valid_hmm]) if np.any(loc_valid_hmm) else np.nan
    mean_loc_hnsw = np.mean(loc_error_hnsw[loc_valid_hnsw]) if np.any(loc_valid_hnsw) else np.nan
    mean_loc_hmm = np.mean(loc_error_hmm[loc_valid_hmm]) if np.any(loc_valid_hmm) else np.nan
    print("Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  HNSW:        Recall@1={r1_hnsw:.4f}  Recall@5={r5_hnsw:.4f}  Recall@10={r10_hnsw:.4f}  Recall@20={r20_hnsw:.4f}  (n={n_eval})")
    print(f"  HNSW + HMM:  Recall@1={r1_hmm:.4f}  Recall@5={r5_hmm:.4f}  Recall@10={r10_hmm:.4f}  Recall@20={r20_hmm:.4f}")
    print("附加指标:")
    print(f"  HNSW:        Center-tile Accuracy@1={center_acc1_hnsw:.4f}  Median localization error={median_loc_hnsw:.2f}px  Mean localization error={mean_loc_hnsw:.2f}px")
    print(f"  HNSW + HMM:  Center-tile Accuracy@1={center_acc1_hmm:.4f}  Median localization error={median_loc_hmm:.2f}px  Mean localization error={mean_loc_hmm:.2f}px")
    print("")
    print("各场景 Recall（仅统计该场景中有坐标法 GT 的 query）:")
    for scene_name, start, end in trajectory_ranges:
        scene_indices = np.arange(start, end)
        scene_valid = valid[scene_indices]
        n_scene_eval = int(np.sum(scene_valid))
        if n_scene_eval == 0:
            print(f"  [{scene_name}] 无有效 GT，跳过")
            continue
        r1_hnsw_scene = correct_1_hnsw[scene_indices][scene_valid].mean()
        r5_hnsw_scene = correct_5_hnsw[scene_indices][scene_valid].mean()
        r10_hnsw_scene = correct_10_hnsw[scene_indices][scene_valid].mean()
        r20_hnsw_scene = correct_20_hnsw[scene_indices][scene_valid].mean()
        r1_hmm_scene = correct_1_hmm[scene_indices][scene_valid].mean()
        r5_hmm_scene = correct_5_hmm[scene_indices][scene_valid].mean()
        r10_hmm_scene = correct_10_hmm[scene_indices][scene_valid].mean()
        r20_hmm_scene = correct_20_hmm[scene_indices][scene_valid].mean()
        scene_center_valid = center_valid[scene_indices]
        scene_loc_valid_hnsw = loc_valid_hnsw[scene_indices]
        scene_loc_valid_hmm = loc_valid_hmm[scene_indices]
        center_hnsw_scene = center_acc_hnsw[scene_indices][scene_center_valid].mean() if np.any(scene_center_valid) else np.nan
        center_hmm_scene = center_acc_hmm[scene_indices][scene_center_valid].mean() if np.any(scene_center_valid) else np.nan
        median_loc_hnsw_scene = np.median(loc_error_hnsw[scene_indices][scene_loc_valid_hnsw]) if np.any(scene_loc_valid_hnsw) else np.nan
        median_loc_hmm_scene = np.median(loc_error_hmm[scene_indices][scene_loc_valid_hmm]) if np.any(scene_loc_valid_hmm) else np.nan
        print(f"  [{scene_name}] HNSW:       R@1={r1_hnsw_scene:.4f}  R@5={r5_hnsw_scene:.4f}  R@10={r10_hnsw_scene:.4f}  R@20={r20_hnsw_scene:.4f}  Center@1={center_hnsw_scene:.4f}  MedErr={median_loc_hnsw_scene:.2f}px  (n={n_scene_eval})")
        print(f"  [{scene_name}] HNSW+HMM:   R@1={r1_hmm_scene:.4f}  R@5={r5_hmm_scene:.4f}  R@10={r10_hmm_scene:.4f}  R@20={r20_hmm_scene:.4f}  Center@1={center_hmm_scene:.4f}  MedErr={median_loc_hmm_scene:.2f}px")
else:
    print("无有效坐标法 GT，请检查 uav_infos.csv 与 GDAL 路径。")

# 将 HNSW 与 HNSW+HMM 的 top-10 结果保存到 ch3/results（每行：查询图像名\t检索到的 db 图像名 x10）
results_dir = Path(_root) / "results"
results_dir.mkdir(parents=True, exist_ok=True)
def _name(i, names): return names[i] if 0 <= i < len(names) else "-"
with open(results_dir / "hnsw_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hnsw[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "hnsw_hmm_top10.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        db_cols = [_name(int(topk_hmm[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "gt_25.txt", "w", encoding="utf-8") as f:
    f.write("query\t" + "\t".join(f"db_{j}" for j in range(1, 26)) + "\n")
    for i in range(n_queries):
        q = _name(i, query_names)
        gt_indices = gt_25_ordered[i][:25] if i < len(gt_25_ordered) and gt_25_ordered[i] else sorted(gt_9_list[i])[:25]
        db_cols = [_name(idx, db_names) for idx in gt_indices]
        db_cols += ["-"] * (25 - len(db_cols))
        f.write("\t".join([q] + db_cols) + "\n")
print(f"按场景检索 Top-10 已保存: {results_dir / 'hnsw_top10.txt'}, {results_dir / 'hnsw_hmm_top10.txt'}")
print(f"GT 已保存: {results_dir / 'gt_25.txt'}")



## 4.1 逐轨迹运行精确检索（Flat 暴力）+ HMM

说明：在每个 query 所属场景内，直接对该场景全部 DB 描述子做内积暴力检索（精确 top-k），再接 WindowViterbiHMM；最终与 HNSW/HNSW+HMM 一起评估。

In [ ]:
# 精确检索（按场景全量暴力）+ HMM
pred_exact = np.full(n_queries, -1, dtype=np.int64)
pred_exact_hmm = np.full(n_queries, -1, dtype=np.int64)

store_topk_exact = min(K, EVAL_TOPK)
topk_exact = np.full((n_queries, store_topk_exact), -1, dtype=np.int64)
topk_exact_hmm = np.full((n_queries, store_topk_exact), -1, dtype=np.int64)

fusion_used_hmm_exact = 0
fusion_fallback_exact = 0
query_idx = 0

# 便于按场景定位 DB 片段
db_scene_lookup = {name: (start, end) for name, start, end in db_scene_ranges}

for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    if scene_name not in db_scene_lookup:
        query_idx += (end - start)
        continue

    db_start, db_end = db_scene_lookup[scene_name]
    if db_end <= db_start:
        query_idx += (end - start)
        continue

    seg_descs = db_descs[db_start:db_end]
    q_descs = query_descs[start:end]
    n_frames = q_descs.shape[0]

    # 与 HNSW 流程一致：从真实轨迹坐标估计位移/速度先验
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])

    query_px = [None] * n_frames
    query_py = [None] * n_frames
    frame_speeds = [None] * n_frames

    n_use = min(len(lats), n_frames)
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = px, py

    for f in range(1, n_use):
        if query_px[f - 1] is None or query_px[f] is None:
            continue
        dx = query_px[f] - query_px[f - 1]
        dy = query_py[f] - query_py[f - 1]
        d_pix = float(np.hypot(dx, dy))
        frame_speeds[f] = d_pix * FPS

    valid_speeds = [s for s in frame_speeds if s is not None and np.isfinite(s) and s > 0]
    traj_mean_speed = float(np.mean(valid_speeds)) if valid_speeds else None

    verbose_hmm = False
    hmm_exact = WindowViterbiHMM(
        coords_for_hmm,
        store_topk_exact,
        uav_speed=traj_mean_speed,
        delta_t=FRAME_INTERVAL,
        window_size=WINDOW_SIZE,
        verbose=verbose_hmm,
    )

    for f in range(n_frames):
        # 场景内暴力内积检索（精确）
        q = q_descs[f]
        scores = seg_descs @ q
        order = np.argsort(-scores)[:store_topk_exact]

        base_inds = (order.astype(np.int64) + db_start)
        base_dists = 1.0 - scores[order].astype(np.float64)

        pred_exact[query_idx] = int(base_inds[0])
        topk_exact[query_idx, :] = -1
        n_top = min(store_topk_exact, len(base_inds))
        topk_exact[query_idx, :n_top] = base_inds[:n_top]

        displacement = None
        frame_speed = frame_speeds[f]
        if f >= 1 and query_px[f - 1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f - 1], query_py[f] - query_py[f - 1])

        hmm_top, _ = hmm_exact.add_frame(
            base_inds,
            base_dists,
            return_top_k=store_topk_exact,
            displacement=displacement,
            frame_speed=frame_speed,
            metadata={"frame_index": f},
            return_debug=True,
        )

        base_ranked = [int(x) for x in base_inds[:store_topk_exact]]
        fused_ranking = [int(x) for x in hmm_top[:store_topk_exact]] if hmm_top else base_ranked.copy()

        warm_start_active = (f < 3 and len(gt_9_list[query_idx]) > 0)
        if USE_HMM_TOP1_FUSION and not warm_start_active and hmm_top:
            base_topn = {int(x) for x in base_inds[: min(FUSION_HNSW_TOPN, len(base_inds))]}
            hmm_top1 = int(hmm_top[0])
            if hmm_top1 in base_topn:
                fusion_used_hmm_exact += 1
                fused_top1 = hmm_top1
            else:
                fusion_fallback_exact += 1
                fused_top1 = int(base_inds[0])

            fused_ranking = [fused_top1]
            fused_ranking.extend(int(x) for x in hmm_top if int(x) != fused_top1)
            fused_ranking.extend(int(x) for x in base_ranked if int(x) != fused_top1)

            deduped = []
            seen = set()
            for idx in fused_ranking:
                if idx in seen:
                    continue
                seen.add(idx)
                deduped.append(idx)
                if len(deduped) >= store_topk_exact:
                    break
            fused_ranking = deduped

        pred_exact_hmm[query_idx] = fused_ranking[0] if fused_ranking else -1
        topk_exact_hmm[query_idx, :] = -1
        for j, idx in enumerate(fused_ranking[:store_topk_exact]):
            topk_exact_hmm[query_idx, j] = int(idx)

        # 与 HNSW 分支保持一致：每场景前 3 帧 warm-start
        if warm_start_active:
            gt_idx = gt_25_ordered[query_idx][0] if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else min(gt_9_list[query_idx])
            pred_exact_hmm[query_idx] = int(gt_idx)
            topk_exact_hmm[query_idx, :] = -1
            topk_exact_hmm[query_idx, 0] = int(gt_idx)
            rest = [x for x in fused_ranking if x != gt_idx][: max(0, store_topk_exact - 1)]
            for j, x in enumerate(rest):
                topk_exact_hmm[query_idx, j + 1] = int(x)
            hmm_exact.override_prev_best(int(gt_idx))

        query_idx += 1

assert query_idx == n_queries
print(
    f"[精确检索+HMM] Top1 融合统计: 使用 HMM top1 {fusion_used_hmm_exact} 次, "
    f"回退到精确检索 top1 {fusion_fallback_exact} 次, "
    f"融合启用={USE_HMM_TOP1_FUSION}, topN={FUSION_HNSW_TOPN}"
)

[精确检索+HMM] Top1 融合统计: 使用 HMM top1 0 次, 回退到精确检索 top1 0 次, 融合启用=False, topN=3


In [ ]:
# 四种方法统一评估：HNSW / HNSW+HMM / 精确检索 / 精确检索+HMM
valid = np.array([len(gt_9_list[i]) > 0 for i in range(n_queries)])
n_eval = int(np.sum(valid))

if n_eval == 0:
    print("无有效坐标法 GT，请检查数据路径。")
else:
    def _recall_stats(topk_arr, top1_arr, gt_sets):
        k1 = min(1, topk_arr.shape[1])
        k5 = min(5, topk_arr.shape[1])
        k10 = min(10, topk_arr.shape[1])
        k20 = min(20, topk_arr.shape[1])

        c1 = np.array([top1_arr[i] in gt_sets[i] if top1_arr[i] >= 0 else False for i in range(len(gt_sets))])
        c5 = np.array([any(topk_arr[i, j] in gt_sets[i] for j in range(k5)) for i in range(len(gt_sets))])
        c10 = np.array([any(topk_arr[i, j] in gt_sets[i] for j in range(k10)) for i in range(len(gt_sets))])
        c20 = np.array([any(topk_arr[i, j] in gt_sets[i] for j in range(k20)) for i in range(len(gt_sets))])

        return (
            float(c1[valid].mean()),
            float(c5[valid].mean()),
            float(c10[valid].mean()),
            float(c20[valid].mean()),
        )

    r_hnsw = _recall_stats(topk_hnsw, pred_hnsw, gt_9_list)
    r_hnsw_hmm = _recall_stats(topk_hmm, pred_hmm, gt_9_list)
    r_exact = _recall_stats(topk_exact, pred_exact, gt_9_list)
    r_exact_hmm = _recall_stats(topk_exact_hmm, pred_exact_hmm, gt_9_list)

    print("Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:")
    print(f"  HNSW:         Recall@1={r_hnsw[0]:.4f}  Recall@5={r_hnsw[1]:.4f}  Recall@10={r_hnsw[2]:.4f}  Recall@20={r_hnsw[3]:.4f}")
    print(f"  HNSW + HMM:   Recall@1={r_hnsw_hmm[0]:.4f}  Recall@5={r_hnsw_hmm[1]:.4f}  Recall@10={r_hnsw_hmm[2]:.4f}  Recall@20={r_hnsw_hmm[3]:.4f}")
    print(f"  精确检索:      Recall@1={r_exact[0]:.4f}  Recall@5={r_exact[1]:.4f}  Recall@10={r_exact[2]:.4f}  Recall@20={r_exact[3]:.4f}")
    print(f"  精确检索 + HMM: Recall@1={r_exact_hmm[0]:.4f}  Recall@5={r_exact_hmm[1]:.4f}  Recall@10={r_exact_hmm[2]:.4f}  Recall@20={r_exact_hmm[3]:.4f}")
    print(f"  (n={n_eval})")

Recall（仅在有坐标法 GT 的 query 上，命中方圆 25 张即正确）:
  HNSW:         Recall@1=0.7406  Recall@5=0.8859  Recall@10=0.9300  Recall@20=0.9641
  HNSW + HMM:   Recall@1=0.7483  Recall@5=0.9003  Recall@10=0.9444  Recall@20=0.9646
  精确检索:      Recall@1=0.7406  Recall@5=0.8858  Recall@10=0.9302  Recall@20=0.9641
  精确检索 + HMM: Recall@1=0.7445  Recall@5=0.9000  Recall@10=0.9439  Recall@20=0.9643
  (n=5988)


## 4b. 全库检索（不按场景）

在**全部 DB 数据**上建一个 HNSW 索引，每条 query 在整库中检索（不限制场景）。

In [ ]:
# HMM 使用的坐标（与按场景检索一致；仅跑本 cell 时也需 n_valid、database_coords 已存在）
use_coords = n_valid > 0
coords_for_hmm = database_coords if use_coords else None

# 复用前面已构建的全库 index（见 HNSW 参数 cell），不重复建索引
index_global = index
print("全库检索使用已构建的 index, K=", K)

k_search = min(K, N_db)
D_global, I_global = index_global.search(query_descs, k_search)
I_global = I_global.astype(np.int64)
dist_global = 1.0 - D_global.astype(np.float64)

pred_hnsw_global = np.full(n_queries, -1, dtype=np.int64)
pred_hmm_global = np.full(n_queries, -1, dtype=np.int64)
store_topk = min(K, EVAL_TOPK)
topk_hnsw_global = np.full((n_queries, store_topk), -1, dtype=np.int64)
topk_hmm_global = np.full((n_queries, store_topk), -1, dtype=np.int64)
fusion_used_hmm_global = 0
fusion_fallback_hnsw_global = 0
query_idx = 0
for traj_idx, (scene_name, start, end) in enumerate(trajectory_ranges):
    n_frames = end - start
    paths = get_scene_paths(scene_name, DATASETS_BASE)
    lats, lons = load_uav_infos(paths["uav_infos_path"])
    gt_tuple, _ = get_geotransform_and_srs(paths["mapbox_path"])
    n_use = min(len(lats), n_frames)
    query_px = [None] * n_frames
    query_py = [None] * n_frames
    frame_speeds = [None] * n_frames
    frame_displacements = []
    for f in range(n_use):
        px, py = lonlat_to_pixel(lons[f], lats[f], gt_tuple)
        query_px[f], query_py[f] = px, py
    for f in range(1, n_use):
        if query_px[f - 1] is None or query_px[f] is None:
            continue
        dx = query_px[f] - query_px[f - 1]
        dy = query_py[f] - query_py[f - 1]
        d_pix = float(np.hypot(dx, dy))
        frame_displacements.append(d_pix)
        frame_speeds[f] = d_pix * FPS
    valid_speeds = [s for s in frame_speeds if s is not None and np.isfinite(s) and s > 0]
    traj_mean_speed = float(np.mean(valid_speeds)) if valid_speeds else None
    if frame_displacements:
        disp_arr = np.asarray(frame_displacements, dtype=np.float64)
        print(
            f"[{scene_name}] 真实帧间像素位移统计: "
            f"mean={disp_arr.mean():.2f}px  median={np.median(disp_arr):.2f}px  "
            f"p90={np.percentile(disp_arr, 90):.2f}px  max={disp_arr.max():.2f}px  (n={len(disp_arr)})"
        )
    else:
        print(f"[{scene_name}] 真实帧间像素位移统计: 无有效位移数据")
    hmm = OnlineHMM(coords_for_hmm, K, uav_speed=traj_mean_speed, delta_t=FRAME_INTERVAL)
    for f in range(n_frames):
        pred_hnsw_global[query_idx] = I_global[query_idx, 0]
        n_top = min(store_topk, I_global.shape[1])
        topk_hnsw_global[query_idx, :n_top] = I_global[query_idx, :n_top]
        inds = I_global[query_idx]
        dists = dist_global[query_idx]
        if len(inds) < K:
            inds = np.concatenate([inds, np.full(K - len(inds), inds[0], dtype=np.int64)])
            dists = np.concatenate([dists, np.full(K - len(dists), dists[0], dtype=np.float64)])
        else:
            inds, dists = inds[:K], dists[:K]
        displacement = None
        frame_speed = frame_speeds[f]
        if f >= 1 and query_px[f-1] is not None and query_px[f] is not None:
            displacement = (query_px[f] - query_px[f-1], query_py[f] - query_py[f-1])
        hmm_top = hmm.update(
            inds,
            dists,
            return_top_k=store_topk,
            displacement=displacement,
            frame_speed=frame_speed,
        )
        hnsw_ranked = [int(x) for x in inds[:store_topk]]
        fused_ranking = [int(x) for x in hmm_top[:store_topk]] if hmm_top else hnsw_ranked.copy()
        warm_start_active = (f < 3 and len(gt_9_list[query_idx]) > 0)
        if USE_HMM_TOP1_FUSION and not warm_start_active and hmm_top:
            hnsw_topn = {int(x) for x in inds[: min(FUSION_HNSW_TOPN, len(inds))]}
            hmm_top1 = int(hmm_top[0])
            if hmm_top1 in hnsw_topn:
                fusion_used_hmm_global += 1
                fused_top1 = hmm_top1
            else:
                fusion_fallback_hnsw_global += 1
                fused_top1 = int(inds[0])
            fused_ranking = [fused_top1]
            fused_ranking.extend(int(x) for x in hmm_top if int(x) != fused_top1)
            fused_ranking.extend(int(x) for x in hnsw_ranked if int(x) != fused_top1)
            deduped_ranking = []
            seen = set()
            for idx in fused_ranking:
                if idx in seen:
                    continue
                seen.add(idx)
                deduped_ranking.append(idx)
                if len(deduped_ranking) >= store_topk:
                    break
            fused_ranking = deduped_ranking
        pred_hmm_global[query_idx] = fused_ranking[0] if fused_ranking else -1
        topk_hmm_global[query_idx, :] = -1
        for j, idx in enumerate(fused_ranking[:store_topk]):
            topk_hmm_global[query_idx, j] = idx
        if warm_start_active:
            gt_idx = gt_25_ordered[query_idx][0] if (query_idx < len(gt_25_ordered) and gt_25_ordered[query_idx]) else min(gt_9_list[query_idx])
            pred_hmm_global[query_idx] = gt_idx
            topk_hmm_global[query_idx, :] = -1
            topk_hmm_global[query_idx, 0] = gt_idx
            rest = [x for x in fused_ranking if x != gt_idx][: max(0, store_topk - 1)]
            for j, x in enumerate(rest):
                topk_hmm_global[query_idx, j + 1] = x
            hmm.override_prev_best(gt_idx)
        query_idx += 1
assert query_idx == n_queries
print(
    f"全库 Top1 融合统计: 使用 HMM top1 {fusion_used_hmm_global} 次, "
    f"回退到 HNSW top1 {fusion_fallback_hnsw_global} 次, "
    f"融合启用={USE_HMM_TOP1_FUSION}, HNSW topN={FUSION_HNSW_TOPN}"
)

topk_eval_5 = min(5, topk_hnsw_global.shape[1])
topk_eval_10 = min(10, topk_hnsw_global.shape[1])
topk_eval_20 = min(20, topk_hnsw_global.shape[1])

c1_h = np.array([topk_hnsw_global[i, 0] in gt_9_list[i] if topk_hnsw_global[i, 0] >= 0 else False for i in range(n_queries)])
c5_h = np.array([any(topk_hnsw_global[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
c10_h = np.array([any(topk_hnsw_global[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])
c20_h = np.array([any(topk_hnsw_global[i, j] in gt_9_list[i] for j in range(topk_eval_20)) for i in range(n_queries)])
c1_m = np.array([pred_hmm_global[i] in gt_9_list[i] for i in range(n_queries)])
c5_m = np.array([any(topk_hmm_global[i, j] in gt_9_list[i] for j in range(topk_eval_5)) for i in range(n_queries)])
c10_m = np.array([any(topk_hmm_global[i, j] in gt_9_list[i] for j in range(topk_eval_10)) for i in range(n_queries)])
c20_m = np.array([any(topk_hmm_global[i, j] in gt_9_list[i] for j in range(topk_eval_20)) for i in range(n_queries)])
if n_eval > 0:
    print("全库检索 Recall（命中方圆 25 张即正确）:")
    print(f"  HNSW:        Recall@1={c1_h[valid].mean():.4f}  Recall@5={c5_h[valid].mean():.4f}  Recall@10={c10_h[valid].mean():.4f}  Recall@20={c20_h[valid].mean():.4f}  (n={n_eval})")
    print(f"  HNSW + HMM:  Recall@1={c1_m[valid].mean():.4f}  Recall@5={c5_m[valid].mean():.4f}  Recall@10={c10_m[valid].mean():.4f}  Recall@20={c20_m[valid].mean():.4f}")
else:
    print("无有效 GT，跳过全库 Recall。")

# 全库检索 top-10 保存到 results（格式同按场景：query\tdb_1\t...\tdb_10）
results_dir = Path(_root) / "results"
results_dir.mkdir(parents=True, exist_ok=True)
def _name_global(i, names): return names[i] if 0 <= i < len(names) else "-"
with open(results_dir / "hnsw_top10_global.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name_global(i, query_names)
        db_cols = [_name_global(int(topk_hnsw_global[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
with open(results_dir / "hnsw_hmm_top10_global.txt", "w", encoding="utf-8") as f:
    f.write("query\tdb_1\tdb_2\tdb_3\tdb_4\tdb_5\tdb_6\tdb_7\tdb_8\tdb_9\tdb_10\n")
    for i in range(n_queries):
        q = _name_global(i, query_names)
        db_cols = [_name_global(int(topk_hmm_global[i, j]), db_names) for j in range(10)]
        f.write("\t".join([q] + db_cols) + "\n")
print(f"全库检索 Top-10 已保存: {results_dir / 'hnsw_top10_global.txt'}, {results_dir / 'hnsw_hmm_top10_global.txt'}")



全库检索使用已构建的 index, K= 20
[city1] 真实帧间像素位移统计: mean=16.24px  median=15.58px  p90=18.64px  max=21.23px  (n=428)
[city2] 真实帧间像素位移统计: mean=13.71px  median=14.04px  p90=15.58px  max=19.52px  (n=327)
[city3] 真实帧间像素位移统计: mean=16.42px  median=15.58px  p90=19.18px  max=22.54px  (n=361)
[industry1] 真实帧间像素位移统计: mean=16.76px  median=17.43px  p90=19.18px  max=22.54px  (n=474)
[industry2] 真实帧间像素位移统计: mean=16.30px  median=15.58px  p90=19.18px  max=21.20px  (n=261)
[industry3] 真实帧间像素位移统计: mean=16.54px  median=15.58px  p90=19.18px  max=22.51px  (n=623)
[park1] 真实帧间像素位移统计: mean=10.89px  median=11.18px  p90=14.04px  max=15.58px  (n=397)
[rural1] 真实帧间像素位移统计: mean=10.92px  median=11.18px  p90=13.51px  max=15.58px  (n=243)
[rural2] 真实帧间像素位移统计: mean=13.60px  median=13.51px  p90=15.58px  max=18.39px  (n=417)
[rural3] 真实帧间像素位移统计: mean=16.01px  median=15.58px  p90=19.18px  max=21.20px  (n=428)
[school] 真实帧间像素位移统计: mean=16.26px  median=15.58px  p90=18.64px  max=20.70px  (n=532)
[suburbs1] 真实帧间像素位移统计: mean=35.37px 